# STAGE 2: Model Selection (Rough Parameter Search)

**Philosophy**: Give each model a **fair chance** by testing a reasonable spread of parameters, not just defaults.

**Goal**: Find the best *representative* performance for each model type before Stage 3 fine-tuning.

**Approach**: 
- Test multiple parameter configurations for each model
- Use RandomizedSearchCV with limited iterations (~10-20)
- Cover enough parameter space to give each model a fair shot
- Avoid exhaustive tuning (that's Stage 3)

**Models to Test**:
1. Logistic Regression (L1, L2, ElasticNet with various C values)
2. Random Forest (varying depth, trees, split criteria)
3. XGBoost (varying learning rates, depths, regularization)
4. Neural Network (varying architectures, regularization)

**Output**: Top 2-3 models for Stage 3

**Note**: Currently using all preprocessed features. After Stage 1 feature engineering is finalized, we can re-run with optimal feature subset.

## Setup and Imports

In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import joblib
import warnings
warnings.filterwarnings('ignore')

# Create results directory if needed
os.makedirs('../results', exist_ok=True)

# Import evaluate_model utility
from evaluate_model import evaluate_model

# Set random seed
np.random.seed(42)

## Load Preprocessed Data

In [2]:
# Load preprocessed data
train_df = pd.read_csv('../data/train_data_preprocessed.csv')
print(f"Training data shape: {train_df.shape}")

# Separate features and target
X = train_df.drop('Revenue', axis=1)
y = train_df['Revenue']

print(f"\nNumber of features: {X.shape[1]}")
print(f"Feature names: {list(X.columns)}")
print(f"\nTarget distribution:")
print(y.value_counts())
print(f"Purchase rate: {y.mean():.1%}")

Training data shape: (9864, 18)

Number of features: 17
Feature names: ['Administrative', 'Administrative_Duration', 'Informational', 'Informational_Duration', 'ProductRelated', 'ProductRelated_Duration', 'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay', 'Month', 'OperatingSystems', 'Browser', 'Region', 'TrafficType', 'VisitorType', 'Weekend']

Target distribution:
Revenue
0    8338
1    1526
Name: count, dtype: int64
Purchase rate: 15.5%


## Setup Cross-Validation

In [3]:
# 5-fold stratified cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Store results
results = []

## Model 1: Logistic Regression (Separate L1, L2, ElasticNet)

In [4]:
# 1a. No Regularization Logistic Regression
print("\n1a. No Regularization Logistic Regression")
print("-" * 40)

lr_none = LogisticRegression(penalty=None, solver='lbfgs', max_iter=1000, random_state=42)


# Just evaluate directly (no hyperparameter search needed)
result_none = evaluate_model(
    lr_none, X, y, cv,
    "Logistic Regression - No Regularization"
)
result_none['best_params'] = "{'penalty': 'none'}"
results.append(result_none)

print(f"CV F1-Score: {result_none['cv_f1']:.4f}")


1a. No Regularization Logistic Regression
----------------------------------------

Evaluating Logistic Regression - No Regularization

5-Fold Cross-Validation Results:
  Accuracy:  0.8806 (+/- 0.0055)
  F1-Score:  0.4897 (+/- 0.0332)
  Precision: 0.7209 (+/- 0.0272)
  Recall:    0.3715 (+/- 0.0338)
  ROC-AUC:   0.8678 (+/- 0.0071)

Full Training Set Performance:
  Accuracy:  0.8830
  F1-Score:  0.5004
  Precision: 0.7372
  Recall:    0.3788
  ROC-AUC:   0.8779

Confusion Matrix:
  TN:  8132  |  FP:   206
  FN:   948  |  TP:   578
CV F1-Score: 0.4897


In [5]:
print("="*80)
print("MODEL 1: Logistic Regression (Testing L1, L2, ElasticNet, No Regularization)")
print("="*80)

# Test each penalty type separately

# 1b. L1 (Lasso) Logistic Regression
print("\n1b. L1 (Lasso) Logistic Regression")
print("-" * 40)
param_grid_l1 = {
    'C': [0.001, 0.01, 0.1, 1.0, 10.0, 100.0],
    'penalty': ['l1'],
    'solver': ['saga', 'liblinear'],
    'max_iter': [1000]
}

lr_l1 = LogisticRegression(random_state=42)
random_search_l1 = RandomizedSearchCV(
    lr_l1, param_grid_l1, n_iter=50, cv=cv, scoring='f1',
    n_jobs=-1, random_state=42, verbose=1
)
random_search_l1.fit(X, y)

print(f"Best C: {random_search_l1.best_params_['C']}")
print(f"Best CV F1-Score: {random_search_l1.best_score_:.4f}")

result_l1 = evaluate_model(
    random_search_l1.best_estimator_, X, y, cv,
    "Logistic Regression - L1 (Lasso)"
)
result_l1['best_params'] = str(random_search_l1.best_params_)
results.append(result_l1)

MODEL 1: Logistic Regression (Testing L1, L2, ElasticNet, No Regularization)

1b. L1 (Lasso) Logistic Regression
----------------------------------------
Fitting 5 folds for each of 12 candidates, totalling 60 fits
Best C: 100.0
Best CV F1-Score: 0.4956

Evaluating Logistic Regression - L1 (Lasso)

5-Fold Cross-Validation Results:
  Accuracy:  0.8833 (+/- 0.0055)
  F1-Score:  0.4956 (+/- 0.0337)
  Precision: 0.7462 (+/- 0.0258)
  Recall:    0.3715 (+/- 0.0329)
  ROC-AUC:   0.8812 (+/- 0.0058)

Full Training Set Performance:
  Accuracy:  0.8836
  F1-Score:  0.4974
  Precision: 0.7493
  Recall:    0.3722
  ROC-AUC:   0.8847

Confusion Matrix:
  TN:  8148  |  FP:   190
  FN:   958  |  TP:   568


In [ ]:
# 1c. L2 (Ridge) Logistic Regression
print("\n1c. L2 (Ridge) Logistic Regression")
print("-" * 40)
param_grid_l2 = {
    'C': [0.001, 0.01, 0.1, 1.0, 10.0, 100.0],
    'penalty': ['l2'],
    'solver': ['saga', 'liblinear'],
    'max_iter': [1000]
}

lr_l2 = LogisticRegression(random_state=42)
random_search_l2 = RandomizedSearchCV(
    lr_l2, param_grid_l2, n_iter=20, cv=cv, scoring='f1',
    n_jobs=-1, random_state=42, verbose=1
)
random_search_l2.fit(X, y)

print(f"Best C: {random_search_l2.best_params_['C']}")
print(f"Best CV F1-Score: {random_search_l2.best_score_:.4f}")

result_l2 = evaluate_model(
    random_search_l2.best_estimator_, X, y, cv,
    "Logistic Regression - L2 (Ridge)"
)
result_l2['best_params'] = str(random_search_l2.best_params_)
results.append(result_l2)



1c. L2 (Ridge) Logistic Regression
----------------------------------------
Fitting 5 folds for each of 6 candidates, totalling 30 fits
Best C: 0.1
Best CV F1-Score: 0.4906

Evaluating Logistic Regression - L2 (Ridge)

5-Fold Cross-Validation Results:
  Accuracy:  0.8815 (+/- 0.0053)
  F1-Score:  0.4906 (+/- 0.0296)
  Precision: 0.7310 (+/- 0.0289)
  Recall:    0.3696 (+/- 0.0280)
  ROC-AUC:   0.8847 (+/- 0.0060)

Full Training Set Performance:
  Accuracy:  0.8822
  F1-Score:  0.4926
  Precision: 0.7382
  Recall:    0.3696
  ROC-AUC:   0.8887

Confusion Matrix:
  TN:  8138  |  FP:   200
  FN:   962  |  TP:   564


In [ ]:
# 1d. ElasticNet Logistic Regression
print("\n1d. ElasticNet Logistic Regression")
print("-" * 40)
param_grid_elastic = {
    'C': [0.001, 0.01, 0.1, 1.0, 10.0, 100.0],
    'penalty': ['elasticnet'],
    'solver': ['saga', 'liblinear'],
    'l1_ratio': [0.3, 0.5, 0.7],
    'max_iter': [1000]
}

lr_elastic = LogisticRegression(random_state=42)
random_search_elastic = RandomizedSearchCV(
    lr_elastic, param_grid_elastic, n_iter=20, cv=cv, scoring='f1',
    n_jobs=-1, random_state=42, verbose=1
)
random_search_elastic.fit(X, y)

print(f"Best C: {random_search_elastic.best_params_['C']}")
print(f"Best l1_ratio: {random_search_elastic.best_params_['l1_ratio']}")
print(f"Best CV F1-Score: {random_search_elastic.best_score_:.4f}")

result_elastic = evaluate_model(
    random_search_elastic.best_estimator_, X, y, cv,
    "Logistic Regression - ElasticNet"
)
result_elastic['best_params'] = str(random_search_elastic.best_params_)
results.append(result_elastic)


1d. ElasticNet Logistic Regression
----------------------------------------
Fitting 5 folds for each of 10 candidates, totalling 50 fits
Best C: 0.01
Best l1_ratio: 0.5
Best CV F1-Score: 0.3080

Evaluating Logistic Regression - ElasticNet

5-Fold Cross-Validation Results:
  Accuracy:  0.8688 (+/- 0.0042)
  F1-Score:  0.3080 (+/- 0.0590)
  Precision: 0.8426 (+/- 0.0549)
  Recall:    0.1920 (+/- 0.0517)
  ROC-AUC:   0.4866 (+/- 0.0402)

Full Training Set Performance:
  Accuracy:  0.8690
  F1-Score:  0.2993
  Precision: 0.8679
  Recall:    0.1809
  ROC-AUC:   0.4775

Confusion Matrix:
  TN:  8296  |  FP:    42
  FN:  1250  |  TP:   276


## Model 2: Random Forest (Varying Depth, Trees, Split Criteria)

In [ ]:
print("\n" + "="*80)
print("MODEL 2: Random Forest")
print("="*80)

# Parameter grid: Varying complexity and regularization
param_grid_rf = {
    'n_estimators': [100, 200],        # basic range for tree count
    'max_depth': [10, 20, None],       # shallow, medium, and unbounded
    'min_samples_split': [2, 5],       # default vs slightly stricter split
    'min_samples_leaf': [1, 2],        # smaller vs slightly larger leaf nodes
    'max_features': ['sqrt', 'log2']   # typical feature sampling options
}

rf_base = RandomForestClassifier(random_state=42, n_jobs=-1)

# Randomized search with 50 iterations
random_search_rf = RandomizedSearchCV(
    rf_base,
    param_grid_rf,
    n_iter=20,
    cv=cv,
    scoring='f1',
    n_jobs=-1,
    random_state=42,
    verbose=1
)

print("\nSearching 20 parameter combinations...")
random_search_rf.fit(X, y)

print(f"\nBest params: {random_search_rf.best_params_}")
print(f"Best CV F1-Score: {random_search_rf.best_score_:.4f}")

# Evaluate best model
result_rf = evaluate_model(
    random_search_rf.best_estimator_, X, y, cv,
    "Random Forest (Best from Search)"
)
result_rf['best_params'] = str(random_search_rf.best_params_)
results.append(result_rf)


MODEL 2: Random Forest

Searching 20 parameter combinations...
Fitting 5 folds for each of 48 candidates, totalling 240 fits

Best params: {'n_estimators': 200, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_features': 'sqrt', 'max_depth': None}
Best CV F1-Score: 0.6554

Evaluating Random Forest (Best from Search)

5-Fold Cross-Validation Results:
  Accuracy:  0.9046 (+/- 0.0041)
  F1-Score:  0.6554 (+/- 0.0188)
  Precision: 0.7426 (+/- 0.0171)
  Recall:    0.5871 (+/- 0.0267)
  ROC-AUC:   0.9277 (+/- 0.0036)

Full Training Set Performance:
  Accuracy:  0.9803
  F1-Score:  0.9327
  Precision: 0.9912
  Recall:    0.8807
  ROC-AUC:   0.9991

Confusion Matrix:
  TN:  8326  |  FP:    12
  FN:   182  |  TP:  1344


## Model 3: XGBoost (Varying Learning Rates, Depths, Regularization)

In [ ]:
print("\n" + "="*80)
print("MODEL 3: XGBoost")
print("="*80)

# Parameter grid: Varying learning rate, depth, and regularization
param_grid_xgb = {
    'n_estimators': [100, 200],          # smaller vs larger ensemble
    'learning_rate': [0.05, 0.1],        # typical conservative & moderate learning rates
    'max_depth': [3, 6, 9],              # shallow, medium, deeper trees
    'subsample': [0.8, 1.0],             # random row sampling
    'colsample_bytree': [0.8, 1.0]       # random feature sampling
}

xgb_base = xgb.XGBClassifier(
    objective='binary:logistic',
    random_state=42,
    n_jobs=-1
)

# Randomized search with 25 iterations
random_search_xgb = RandomizedSearchCV(
    xgb_base,
    param_grid_xgb,
    n_iter=20,
    cv=cv,
    scoring='f1',
    n_jobs=-1,
    random_state=42,
    verbose=1
)

print("\nSearching 25 parameter combinations...")
random_search_xgb.fit(X, y)

print(f"\nBest params: {random_search_xgb.best_params_}")
print(f"Best CV F1-Score: {random_search_xgb.best_score_:.4f}")

# Evaluate best model
result_xgb = evaluate_model(
    random_search_xgb.best_estimator_, X, y, cv,
    "XGBoost (Best from Search)"
)
result_xgb['best_params'] = str(random_search_xgb.best_params_)
results.append(result_xgb)


MODEL 3: XGBoost

Searching 25 parameter combinations...
Fitting 5 folds for each of 48 candidates, totalling 240 fits

Best params: {'subsample': 1.0, 'n_estimators': 200, 'max_depth': 3, 'learning_rate': 0.05, 'colsample_bytree': 0.8}
Best CV F1-Score: 0.6635

Evaluating XGBoost (Best from Search)

5-Fold Cross-Validation Results:
  Accuracy:  0.9046 (+/- 0.0038)
  F1-Score:  0.6635 (+/- 0.0190)
  Precision: 0.7296 (+/- 0.0094)
  Recall:    0.6088 (+/- 0.0275)
  ROC-AUC:   0.9322 (+/- 0.0049)

Full Training Set Performance:
  Accuracy:  0.9163
  F1-Score:  0.7029
  Precision: 0.7791
  Recall:    0.6402
  ROC-AUC:   0.9475

Confusion Matrix:
  TN:  8061  |  FP:   277
  FN:   549  |  TP:   977


## Model 4: Neural Network (Varying Architectures, Regularization)

In [ ]:
print("\n" + "="*80)
print("MODEL 4: Neural Network")
print("="*80)

# Neural network needs scaling
nn_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', MLPClassifier(
        max_iter=300,
        early_stopping=True,
        validation_fraction=0.2,
        random_state=42
    ))
])

# Parameter grid: Varying architecture and regularization
param_grid_nn = {
    'classifier__hidden_layer_sizes': [
        (32,), (64,), (128,),
        (64, 32), (128, 64), (128, 64, 32),
        (256, 128, 64)
    ],
    'classifier__alpha': [0.0001, 0.001, 0.01, 0.1],  # L2 regularization
    'classifier__learning_rate_init': [0.0001, 0.001, 0.01],
    'classifier__activation': ['relu', 'tanh']
}

# Randomized search with 50 iterations
random_search_nn = RandomizedSearchCV(
    nn_pipeline,
    param_grid_nn,
    n_iter=50,
    cv=cv,
    scoring='f1',
    n_jobs=-1,
    random_state=42,
    verbose=1
)

print("\nSearching 20 parameter combinations...")
random_search_nn.fit(X, y)

print(f"\nBest params: {random_search_nn.best_params_}")
print(f"Best CV F1-Score: {random_search_nn.best_score_:.4f}")

# Evaluate best model
result_nn = evaluate_model(
    random_search_nn.best_estimator_, X, y, cv,
    "Neural Network (Best from Search)"
)
result_nn['best_params'] = str(random_search_nn.best_params_)
results.append(result_nn)


MODEL 4: Neural Network

Searching 20 parameter combinations...
Fitting 5 folds for each of 50 candidates, totalling 250 fits

Best params: {'classifier__learning_rate_init': 0.001, 'classifier__hidden_layer_sizes': (256, 128, 64), 'classifier__alpha': 0.1, 'classifier__activation': 'relu'}
Best CV F1-Score: 0.6299

Evaluating Neural Network (Best from Search)

5-Fold Cross-Validation Results:
  Accuracy:  0.8972 (+/- 0.0061)
  F1-Score:  0.6299 (+/- 0.0301)
  Precision: 0.7107 (+/- 0.0260)
  Recall:    0.5674 (+/- 0.0462)
  ROC-AUC:   0.9026 (+/- 0.0045)

Full Training Set Performance:
  Accuracy:  0.9043
  F1-Score:  0.6612
  Precision: 0.7310
  Recall:    0.6035
  ROC-AUC:   0.9293

Confusion Matrix:
  TN:  7999  |  FP:   339
  FN:   605  |  TP:   921


## Compare All Models

In [11]:
# Create comparison table
results_df = pd.DataFrame(results)

# Sort by F1-Score
results_df = results_df.sort_values('cv_f1', ascending=False).reset_index(drop=True)

print("\n" + "="*100)
print("STAGE 2: MODEL SELECTION RESULTS (Sorted by F1-Score)")
print("="*100)
print(f"\n{'Rank':<6} {'Model':<35} {'F1':<12} {'ROC-AUC':<12} {'Precision':<12} {'Recall':<12}")
print("-" * 100)

for idx, row in results_df.iterrows():
    print(f"{idx+1:<6} {row['model_name']:<35} {row['cv_f1']:<12.4f} {row['cv_roc_auc']:<12.4f} {row['cv_precision']:<12.4f} {row['cv_recall']:<12.4f}")

# Select top 2-3 models
top_k = min(3, len(results_df))
top_models = results_df.head(top_k)

print("\n" + "="*100)
print(f"TOP {top_k} MODELS SELECTED FOR STAGE 3 (Fine-Tuning)")
print("="*100)
for idx, row in top_models.iterrows():
    print(f"\n{idx+1}. {row['model_name']}")
    print(f"   F1-Score:  {row['cv_f1']:.4f}")
    print(f"   ROC-AUC:   {row['cv_roc_auc']:.4f}")
    print(f"   Best params from rough search: {row['best_params']}")


STAGE 2: MODEL SELECTION RESULTS (Sorted by F1-Score)

Rank   Model                               F1           ROC-AUC      Precision    Recall      
----------------------------------------------------------------------------------------------------
1      XGBoost (Best from Search)          0.6635       0.9322       0.7296       0.6088      
2      Random Forest (Best from Search)    0.6554       0.9277       0.7426       0.5871      
3      Neural Network (Best from Search)   0.6299       0.9026       0.7107       0.5674      
4      Logistic Regression - L1 (Lasso)    0.4956       0.8812       0.7462       0.3715      
5      Logistic Regression - L2 (Ridge)    0.4906       0.8847       0.7310       0.3696      
6      Logistic Regression - No Regularization 0.4897       0.8678       0.7209       0.3715      
7      Logistic Regression - ElasticNet    0.3080       0.4866       0.8426       0.1920      

TOP 3 MODELS SELECTED FOR STAGE 3 (Fine-Tuning)

1. XGBoost (Best from Search)

## Save Results for Stage 3

In [12]:
# Save comparison table
results_df.to_csv('../results/stage2_model_comparison.csv', index=False)
print("\nResults saved to: results/stage2_model_comparison.csv")

# Save top models list
top_models_list = top_models['model_name'].tolist()
with open('../results/stage2_top_models.txt', 'w') as f:
    f.write("\n".join(top_models_list))
print("Top models list saved to: results/stage2_top_models.txt")


Results saved to: results/stage2_model_comparison.csv
Top models list saved to: results/stage2_top_models.txt


## Summary

**STAGE 2 COMPLETE!** ✅

**What We Did**:
- Gave each model a **fair chance** by testing multiple parameter configurations
- Used RandomizedSearchCV with limited iterations (15-25) to cover parameter space
- Tested key variations:
  - **Logistic Regression**: L1, L2, ElasticNet with various C values
  - **Random Forest**: Depth, trees, split criteria, feature sampling
  - **XGBoost**: Learning rates, depths, L1/L2 regularization
  - **Neural Network**: Architectures, alpha values, learning rates

**Key Insight**: 
Each model was evaluated at its best rough configuration, not just defaults. This ensures we're comparing the true potential of each model type.

**Next Step**: 
Move to Stage 3 to fine-tune ONLY the top 2-3 models with more exhaustive search.

**Outputs**:
- `results/stage2_model_comparison.csv` - All model results with best params
- `results/stage2_top_models.txt` - Top models list for Stage 3